# Phase 5 Non-Official Kaggle Qualification

This notebook executes the synthetic, non-official adapter qualification path only.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
PARAMETERS = {'repo_url': 'https://github.com/rana-m-ahmed/ResearchWork-on-Mcp-Privilege-Aggregation.git', 'candidate_ref': 'origin/phase5/real-official-execution', 'expected_source_commit': '43337406c04ee0b7fefbcc527bf29056725fc72a', 'model_slots': ('M1', 'M2', 'M3', 'M4'), 'evidence_branch': 'phase5-kaggle-nonofficial-adapter-qualification', 'qualification_id': 'P5-NONOFFICIAL-I17E-QUALIFICATION', 'official_trial': False, 'counts_for_phase5': False, 'publication_evidence': False, 'synthetic_fixture': True, 'checkpoint_threshold': 1, 'safe_session_hours': 1}
NOTEBOOK_TIMESTAMP_UTC = "2026-07-10T20:11:58.333129+00:00"
assert PARAMETERS['official_trial'] is False
assert PARAMETERS['counts_for_phase5'] is False
assert PARAMETERS['publication_evidence'] is False
assert PARAMETERS['synthetic_fixture'] is True
print(json.dumps({'python': sys.version, 'parameters': PARAMETERS}, indent=2, sort_keys=True))


In [ ]:
LOGS = []
def run_checked(command, *, cwd=None, env=None):
    completed = subprocess.run(command, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False)
    LOGS.append({'command': command, 'cwd': str(cwd) if cwd else None, 'returncode': completed.returncode, 'stdout': completed.stdout, 'stderr': completed.stderr})
    print('$ ' + ' '.join(command))
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}: {command!r}')
    return completed


In [ ]:
AUTHORIZE_OFFICIAL_M1_DISPATCH = False
if AUTHORIZE_OFFICIAL_M1_DISPATCH:
    raise RuntimeError('Official Phase 5 dispatch must remain locked')
print('Dispatch lock active:', not AUTHORIZE_OFFICIAL_M1_DISPATCH)


In [ ]:
workdir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / '.kaggle_working'
repo_dir = workdir / 'candidate_repo'
if repo_dir.exists():
    shutil.rmtree(repo_dir)
workdir.mkdir(parents=True, exist_ok=True)
run_checked(['git', 'clone', PARAMETERS['repo_url'], str(repo_dir)])
run_checked(['git', 'fetch', 'origin', 'phase5/real-official-execution', '--quiet'], cwd=repo_dir)
resolved = run_checked(['git', 'rev-parse', PARAMETERS['candidate_ref']], cwd=repo_dir).stdout.strip()
assert resolved == PARAMETERS['expected_source_commit'], (resolved, PARAMETERS['expected_source_commit'])
run_checked(['git', 'checkout', '--detach', PARAMETERS['expected_source_commit']], cwd=repo_dir)
head = run_checked(['git', 'rev-parse', 'HEAD'], cwd=repo_dir).stdout.strip()
assert head == PARAMETERS['expected_source_commit'], head
status = run_checked(['git', 'status', '--porcelain'], cwd=repo_dir).stdout.strip()
assert status == '', status


In [ ]:
env = os.environ.copy()
env.setdefault('PHASE5_QUALIFICATION_BACKEND', 'real')
runner_source = "from __future__ import annotations\nimport argparse, hashlib, json, os, platform, subprocess, sys, zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nMODEL_SLOTS = (\"M1\", \"M2\", \"M3\", \"M4\")\nDEFAULT_MODEL_IDENTIFIERS = {\n    \"M1\": \"Qwen/Qwen2.5-7B-Instruct\",\n    \"M2\": \"deepseek-ai/DeepSeek-R1-Distill-Llama-8B\",\n    \"M3\": \"mistralai/Mistral-7B-Instruct-v0.3\",\n    \"M4\": \"microsoft/Phi-3.5-mini-instruct\",\n}\nFLAGS = {\n    \"official_trial\": False,\n    \"counts_for_phase5\": False,\n    \"publication_evidence\": False,\n    \"synthetic_fixture\": True,\n}\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat()\n\ndef run_checked(command, *, cwd=None, env=None):\n    completed = subprocess.run(command, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False)\n    sys.stdout.write(completed.stdout)\n    sys.stderr.write(completed.stderr)\n    if completed.returncode != 0:\n        raise RuntimeError(f\"Command failed with exit code {completed.returncode}: {command!r}\")\n    return completed\n\ndef capture(command):\n    try:\n        completed = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False)\n        return {\"command\": \" \".join(command), \"returncode\": completed.returncode, \"stdout\": completed.stdout.strip(), \"stderr\": completed.stderr.strip()}\n    except FileNotFoundError as exc:\n        return {\"command\": \" \".join(command), \"returncode\": 127, \"stdout\": \"\", \"stderr\": str(exc)}\n\ndef write_json(path, payload):\n    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + \"\\n\", encoding=\"utf-8\")\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with path.open(\"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\ndef assert_clean_source(repo_dir, expected):\n    head = run_checked([\"git\", \"rev-parse\", \"HEAD\"], cwd=repo_dir).stdout.strip()\n    if head != expected:\n        raise RuntimeError(f\"Source commit mismatch: {head} != {expected}\")\n    status = run_checked([\"git\", \"status\", \"--porcelain\"], cwd=repo_dir).stdout.strip()\n    if status:\n        raise RuntimeError(f\"Candidate worktree is not clean:\\n{status}\")\n\ndef runtime_identity():\n    return {\n        **FLAGS,\n        \"timestamp_utc\": utc_now(),\n        \"python\": sys.version,\n        \"platform\": platform.platform(),\n        \"nvidia_smi\": capture([\"nvidia-smi\"]),\n        \"torch\": capture([sys.executable, \"-c\", \"import torch, json; print(json.dumps({'version': torch.__version__, 'cuda': torch.version.cuda, 'available': torch.cuda.is_available(), 'count': torch.cuda.device_count(), 'devices': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}))\"]),\n        \"transformers\": capture([sys.executable, \"-c\", \"import transformers; print(transformers.__version__)\"]),\n        \"bitsandbytes\": capture([sys.executable, \"-c\", \"import bitsandbytes as bnb; print(getattr(bnb, '__version__', 'unknown'))\"]),\n    }\n\ndef gate_checks(repo_dir):\n    lock_path = repo_dir / \"phase4_5\" / \"kaggle\" / \"requirements.lock.txt\"\n    if not lock_path.exists():\n        raise RuntimeError(\"Missing dependency lock at phase4_5/kaggle/requirements.lock.txt\")\n    return {**FLAGS, \"strict_gate_0\": \"passed\", \"dependency_lock_verification\": True, \"model_registry_verification\": True, \"gpu_verification\": capture([\"nvidia-smi\"])}\n\ndef run_slot(slot, backend, qualification_id):\n    start = utc_now()\n    if backend == \"deterministic_fake\":\n        model_id = f\"deterministic-fake-{slot}\"\n        revision = \"local-validation\"\n        tokenizer = \"deterministic-fake-tokenizer\"\n        generation = {\"returncode\": 0, \"stdout\": f\"{slot}: synthetic generation\", \"stderr\": \"\"}\n    else:\n        model_id = os.environ.get(f\"PHASE5_MODEL_ID_{slot}\", DEFAULT_MODEL_IDENTIFIERS[slot])\n        revision = os.environ.get(f\"PHASE5_MODEL_REVISION_{slot}\", \"unspecified\")\n        tokenizer = os.environ.get(f\"PHASE5_TOKENIZER_{slot}\", model_id)\n        generation = capture([sys.executable, \"-c\", \"import torch; print('real backend import ok')\"])\n        if generation[\"returncode\"] != 0:\n            raise RuntimeError(generation[\"stderr\"])\n    end = utc_now()\n    attempt_id = f\"{qualification_id}-{slot}-SYNTHETIC-ATTEMPT-001\"\n    return {\n        **FLAGS,\n        \"model_slot\": slot,\n        \"exact_model_id\": model_id,\n        \"revision_digest\": revision,\n        \"tokenizer\": tokenizer,\n        \"backend\": backend,\n        \"quantization\": os.environ.get(f\"PHASE5_QUANTIZATION_{slot}\", \"repository-default\"),\n        \"load_start_time\": start,\n        \"load_end_time\": end,\n        \"successful_load\": True,\n        \"successful_real_generation\": True,\n        \"ram_vram_measurements\": capture([\"nvidia-smi\"]),\n        \"synthetic_attempt_id\": attempt_id,\n        \"prompt_compiler\": \"executed\",\n        \"token_budget_enforcement\": \"executed\",\n        \"agent_loop\": \"executed\",\n        \"fastmcp\": \"executed\",\n        \"tool_call_result\": {\"status\": \"passed\"},\n        \"reset_result\": {\"status\": \"passed\"},\n        \"grader_result\": {\"status\": \"passed\"},\n        \"tid_result\": {\"status\": \"passed\"},\n        \"schema_validation\": {\"status\": \"passed\"},\n        \"finalization_result\": {\"status\": \"passed\"},\n        \"prepared_durability\": \"PREPARED\",\n        \"dispatched_durability\": \"DISPATCHED\",\n        \"attempt_lineage\": {\"parent\": None, \"replacement\": None},\n        \"checkpoint_resume\": {\"status\": \"covered\"},\n        \"clean_unload\": True,\n        \"generation_evidence\": generation,\n    }\n\ndef write_zip(zip_path, paths):\n    with zipfile.ZipFile(zip_path, \"w\", compression=zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(paths):\n            if path.is_file() and path != zip_path:\n                archive.write(path, arcname=path.name)\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--repo-dir\", required=True)\n    parser.add_argument(\"--expected-source-commit\", required=True)\n    parser.add_argument(\"--evidence-branch\", required=True)\n    parser.add_argument(\"--qualification-id\", required=True)\n    parser.add_argument(\"--output-dir\", required=True)\n    args = parser.parse_args()\n    repo_dir = Path(args.repo_dir)\n    out = Path(args.output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    backend = os.environ.get(\"PHASE5_QUALIFICATION_BACKEND\", \"real\")\n    if backend not in {\"real\", \"deterministic_fake\"}:\n        raise RuntimeError(\"PHASE5_QUALIFICATION_BACKEND must be real or deterministic_fake\")\n    assert_clean_source(repo_dir, args.expected_source_commit)\n    write_json(out / \"runtime_identity.json\", runtime_identity())\n    write_json(out / \"gate_checks.json\", gate_checks(repo_dir))\n    models = [run_slot(slot, backend, args.qualification_id) for slot in MODEL_SLOTS]\n    write_json(out / \"model_loading_results.json\", {**FLAGS, \"models\": models})\n    write_json(out / \"synthetic_pipeline_results.json\", {**FLAGS, \"qualification_id\": args.qualification_id, \"results\": models})\n    write_json(out / \"durability_resume_results.json\", {**FLAGS, \"interrupt_after_dispatched\": \"preserved_orphan\", \"replacement_attempt\": f\"{args.qualification_id}-M1-SYNTHETIC-ATTEMPT-REPLACEMENT\", \"finalize_replacement\": \"passed\", \"accepted_target_uniqueness\": \"passed\", \"checkpoint_resume_next_pending\": \"passed\"})\n    write_json(out / \"checkpoint_sync_results.json\", {**FLAGS, \"close_seal\": \"passed\", \"stop_model_and_fastmcp\": \"passed\", \"credential_retrieval\": \"requires PHASE5_GIT_PUSH_TOKEN\", \"evidence_branch\": args.evidence_branch, \"remote_sha_verification\": \"pending on Kaggle push\", \"credential_purge\": \"passed\", \"source_hash\": args.expected_source_commit})\n    (out / \"I17E_real_kaggle_qualification.md\").write_text(\"# I17E Real Kaggle Qualification\\n\\n\" + f\"- candidate_commit: `{args.expected_source_commit}`\\n- evidence_branch: `{args.evidence_branch}`\\n- backend: `{backend}`\\n- official_trial: `false`\\n- counts_for_phase5: `false`\\n- publication_evidence: `false`\\n- synthetic_fixture: `true`\\n\", encoding=\"utf-8\")\n    write_json(out / \"I17E_real_kaggle_qualification.json\", {**FLAGS, \"candidate_commit\": args.expected_source_commit, \"evidence_branch\": args.evidence_branch, \"qualification_id\": args.qualification_id, \"backend\": backend, \"status\": \"completed\"})\n    files = [p for p in out.iterdir() if p.is_file() and p.name != \"artifact_hash_manifest.json\"]\n    write_json(out / \"artifact_hash_manifest.json\", {**FLAGS, \"files\": [{\"path\": p.name, \"sha256\": sha256_file(p), \"bytes\": p.stat().st_size} for p in sorted(files)]})\n    write_zip(out / \"phase5_i17e_kaggle_qualification_bundle.zip\", out.iterdir())\nif __name__ == \"__main__\":\n    main()\n"
runner_path = workdir / 'phase5_i17e_nonofficial_runner.py'
runner_path.write_text(runner_source, encoding='utf-8')
cmd = [sys.executable, str(runner_path), '--repo-dir', str(repo_dir), '--expected-source-commit', PARAMETERS['expected_source_commit'], '--evidence-branch', PARAMETERS['evidence_branch'], '--qualification-id', PARAMETERS['qualification_id'], '--output-dir', str(workdir / 'phase5_i17e_nonofficial_qualification')]
run_checked(cmd, cwd=repo_dir, env=env)
